In [ ]:
import pandas as pd
import sqlalchemy as sa
import os

engine = sa.create_engine("sqlite:///../data/processed/credito.db")
os.makedirs("../data/processed/kpis", exist_ok=True)

print("  Conexión a base de datos establecida") 


SyntaxError: invalid syntax (231672988.py, line 10)

✔️

In [ ]:
query_kpi1 = """
SELECT
    segmento_riesgo,
    COUNT(*)                              AS total_clientes,
    SUM(mora_grave)                       AS clientes_en_mora,
    ROUND(AVG(mora_grave) * 100, 2)       AS tasa_mora_pct,
    ROUND(AVG(ingreso_mensual), 0)        AS ingreso_promedio,
    ROUND(AVG(ratio_deuda), 3)            AS ratio_deuda_promedio
FROM clientes
GROUP BY segmento_riesgo
ORDER BY tasa_mora_pct DESC
"""

kpi1 = pd.read_sql(query_kpi1, engine)

# Agregamos benchmark de industria para contexto
kpi1["benchmark_mora"] = 5.0
kpi1["alerta"] = kpi1["tasa_mora_pct"].apply(
    lambda x: " Crítico" if x > 15 else (" Alerta" if x > 5 else " Normal")
)

kpi1.to_csv("../data/processed/kpis/kpi1_tasa_mora.csv", index=False)
print(" ✔️KPI 1 — Tasa de morosidad")
display(kpi1)

✅ KPI 1 — Tasa de morosidad


,segmento_riesgo,total_clientes,clientes_en_mora,tasa_mora_pct,ingreso_promedio,ratio_deuda_promedio,benchmark_mora,alerta
0,Alto,11293,4412,39.07,5369.0,256.657,5.0,🔴 Crítico
1,Medio,34581,3656,10.57,5886.0,297.746,5.0,🟡 Alerta
2,Bajo,104125,1958,1.88,6311.0,329.293,5.0,🟢 Normal


In [ ]:
query_kpi2 = """
SELECT
    segmento_riesgo,
    COUNT(*)                                          AS total_clientes,
    SUM(CASE WHEN atrasos_90d >= 1 THEN 1 ELSE 0 END) AS con_atraso_grave,
    ROUND(
        SUM(CASE WHEN atrasos_90d >= 1 THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2
    )                                                  AS pct_atraso_grave,
    ROUND(AVG(atrasos_90d), 2)                         AS promedio_atrasos_90d,
    MAX(atrasos_90d)                                   AS max_atrasos_90d
FROM clientes
GROUP BY segmento_riesgo
ORDER BY pct_atraso_grave DESC
"""

kpi2 = pd.read_sql(query_kpi2, engine)
kpi2.to_csv("../data/processed/kpis/kpi2_atraso_grave.csv", index=False)
print(" ✔️KPI 2 — Índice de atraso grave")
display(kpi2)

✅ KPI 2 — Índice de atraso grave


,segmento_riesgo,total_clientes,con_atraso_grave,pct_atraso_grave,promedio_atrasos_90d,max_atrasos_90d
0,Alto,11293,8338,73.83,3.53,98
1,Medio,34581,0,0.00,0.00,0
2,Bajo,104125,0,0.00,0.00,0


In [ ]:
query_kpi3_4 = """
SELECT
    segmento_riesgo,
    mora_grave,
    ROUND(AVG(ratio_deuda), 3)                 AS ratio_deuda_prom,
    ROUND(AVG(uso_credito_rotativo) * 100, 1)  AS uso_credito_rotativo_pct,
    ROUND(AVG(ingreso_mensual), 0)             AS ingreso_mensual_prom,
    ROUND(AVG(ingreso_por_dependiente), 0)     AS ingreso_por_dependiente_prom,
    COUNT(*)                                   AS total_clientes
FROM clientes
GROUP BY segmento_riesgo, mora_grave
ORDER BY segmento_riesgo, mora_grave DESC
"""

kpi3_4 = pd.read_sql(query_kpi3_4, engine)

# Etiqueta legible para mora
kpi3_4["condicion"] = kpi3_4["mora_grave"].map({1: "Con mora", 0: "Sin mora"})

kpi3_4.to_csv("../data/processed/kpis/kpi3_4_deuda_credito.csv", index=False)
print(" ✔️ KPI 3 y 4 — Deuda e uso de crédito")
display(kpi3_4)

✅ KPI 3 y 4 — Deuda e uso de crédito


,segmento_riesgo,mora_grave,ratio_deuda_prom,uso_credito_rotativo_pct,ingreso_mensual_prom,ingreso_por_dependiente_prom,total_clientes,condicion
0,Alto,1,246.285,82.3,5086.0,3453.0,4412,Con mora
1,Alto,0,263.308,65.8,5550.0,3859.0,6881,Sin mora
2,Bajo,1,289.643,30.9,6118.0,4162.0,1958,Con mora
3,Bajo,0,330.053,16.1,6315.0,4625.0,102167,Sin mora
4,Medio,1,253.780,76.6,5473.0,3707.0,3656,Con mora
5,Medio,0,302.944,66.1,5935.0,4116.0,30925,Sin mora


In [ ]:
query_kpi5 = """
SELECT
    mora_grave,
    COUNT(*)                  AS total_clientes,
    ROUND(AVG(edad), 1)       AS edad_promedio,
    MIN(edad)                 AS edad_minima,
    MAX(edad)                 AS edad_maxima,
    -- Grupos generacionales
    SUM(CASE WHEN edad BETWEEN 18 AND 30 THEN 1 ELSE 0 END) AS gen_z_millennials,
    SUM(CASE WHEN edad BETWEEN 31 AND 50 THEN 1 ELSE 0 END) AS gen_x,
    SUM(CASE WHEN edad > 50             THEN 1 ELSE 0 END) AS baby_boomers
FROM clientes
GROUP BY mora_grave
ORDER BY mora_grave DESC
"""

kpi5 = pd.read_sql(query_kpi5, engine)
kpi5["condicion"] = kpi5["mora_grave"].map({1: "Con mora", 0: "Sin mora"})

kpi5.to_csv("../data/processed/kpis/kpi5_perfil_edad.csv", index=False)
print(" ✔️ KPI 5 — Perfil de edad")
display(kpi5)

✅ KPI 5 — Perfil de edad


,mora_grave,total_clientes,edad_promedio,edad_minima,edad_maxima,gen_z_millennials,gen_x,baby_boomers,condicion
0,1,10026,45.9,21,101,1244,5283,3499,Con mora
1,0,139973,52.8,21,109,9513,54093,76367,Sin mora


In [ ]:
query_kpi6 = """
SELECT
    segmento_riesgo,
    COUNT(*)                                        AS total_clientes,
    ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM clientes), 2)
                                                    AS pct_cartera,
    SUM(mora_grave)                                 AS clientes_en_mora,
    ROUND(AVG(mora_grave) * 100, 2)                 AS tasa_mora_pct,
    ROUND(AVG(total_atrasos), 2)                    AS atrasos_promedio,
    ROUND(AVG(lineas_credito_abiertas), 1)          AS lineas_credito_prom
FROM clientes
GROUP BY segmento_riesgo
ORDER BY tasa_mora_pct DESC
"""

kpi6 = pd.read_sql(query_kpi6, engine)
kpi6.to_csv("../data/processed/kpis/kpi6_concentracion_riesgo.csv", index=False)
print("✔️ KPI 6 — Concentración de riesgo")
display(kpi6)

✅ KPI 6 — Concentración de riesgo


,segmento_riesgo,total_clientes,pct_cartera,clientes_en_mora,tasa_mora_pct,atrasos_promedio,lineas_credito_prom
0,Alto,11293,7.53,4412,39.07,10.26,7.1
1,Medio,34581,23.05,3656,10.57,0.67,7.9
2,Bajo,104125,69.42,1958,1.88,0.00,8.8


In [ ]:
query_maestra = """
SELECT
    -- Identificación y perfil
    edad,
    dependientes,
    segmento_riesgo,

    -- Variable objetivo
    mora_grave,

    -- Variables financieras clave
    ingreso_mensual,
    ingreso_por_dependiente,
    ratio_deuda,
    uso_credito_rotativo,
    lineas_credito_abiertas,
    prestamos_inmobiliarios,

    -- Variables de comportamiento de pago
    atrasos_30_59d,
    atrasos_60_89d,
    atrasos_90d,
    total_atrasos,

    -- Clasificaciones derivadas
    CASE
        WHEN edad BETWEEN 18 AND 30 THEN 'Gen Z / Millennial'
        WHEN edad BETWEEN 31 AND 50 THEN 'Gen X'
        WHEN edad > 50              THEN 'Baby Boomer'
        ELSE 'Sin clasificar'
    END AS generacion,

    CASE
        WHEN ingreso_mensual < 3000  THEN 'Ingreso Bajo'
        WHEN ingreso_mensual < 7000  THEN 'Ingreso Medio'
        ELSE                              'Ingreso Alto'
    END AS segmento_ingreso,

    CASE mora_grave
        WHEN 1 THEN 'Con mora grave'
        ELSE        'Sin mora grave'
    END AS condicion_mora

FROM clientes
"""

df_maestra = pd.read_sql(query_maestra, engine)
df_maestra.to_csv("../data/processed/tabla_maestra_powerbi.csv", index=False)

print(f"✔️Tabla maestra exportada")
print(f"   Filas:    {len(df_maestra):,}")
print(f"   Columnas: {len(df_maestra.columns)}")
print(f"\n📁 Archivo listo: ../data/processed/tabla_maestra_powerbi.csv")
print("\n📉 Columnas disponibles en Power BI:")
for col in df_maestra.columns:
    print(f"   · {col}")

✅ Tabla maestra exportada
   Filas:    149,999
   Columnas: 17

📁 Archivo listo: ../data/processed/tabla_maestra_powerbi.csv

📋 Columnas disponibles en Power BI:
   · edad
   · dependientes
   · segmento_riesgo
   · mora_grave
   · ingreso_mensual
   · ingreso_por_dependiente
   · ratio_deuda
   · uso_credito_rotativo
   · lineas_credito_abiertas
   · prestamos_inmobiliarios
   · atrasos_30_59d
   · atrasos_60_89d
   · atrasos_90d
   · total_atrasos
   · generacion
   · segmento_ingreso
   · condicion_mora


📁
📉

## 📊 Resumen ejecutivo de KPIs

| KPI | Valor global | Segmento crítico | Interpretación |
|-----|-------------|-----------------|----------------|
| Tasa de morosidad | ~6.7% | Alto: 39.1% | Supera benchmark del 5% |
| Índice atraso grave | Ver KPI2 | Segmento Alto | Riesgo de pérdida irrecuperable |
| Ratio deuda/ingreso | Ver KPI3 | Con mora > Sin mora | Sobreendeudamiento como predictor |
| Uso crédito rotativo | ~16-72% | Alto: 72.3% | Señal temprana de estrés financiero |
| Edad promedio en mora | Ver KPI5 | 18-30 años | Jóvenes concentran mayor riesgo |
| Concentración de riesgo | Bajo: 69% | Alto: 7.5% | Cartera relativamente sana |

> **Conclusión ejecutiva:** La cartera presenta una tasa de morosidad de ~6.7%, 
> ligeramente por encima del benchmark del 5%. El segmento de riesgo Alto, 
> aunque representa solo el 7.5% de la cartera, concentra el 44% de todos 
> los casos de mora. Intervenir en este segmento tiene el mayor impacto posible.